In [ ]:
import os
import dill
import shutil
import pandas as pd
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
import numpy as np
import random

import warnings

warnings.filterwarnings("ignore")

In [ ]:
all_df = pd.read_csv('/MIMIC_IV/Median_B/All_TS_non_49023.csv')
print(all_df.shape)

(49023, 170)


In [ ]:
diags = pd.read_csv('/MIMIC_IV/Median_B/All_diags_binary.csv')
diag_col = diags.columns
diag_col = diag_col[~diag_col.isin(['hadm_id'])].tolist()

In [ ]:
chart_col = []
lab_col = []
proc_col = []
med_col = []

for item in all_df.columns.tolist():
    if item.startswith('CHART'):
        chart_col.append(item)
    elif item.startswith('LAB'):
        lab_col.append(item)
    elif item.startswith('PROC'):
        proc_col.append(item)
    elif item.startswith('MED'):
        med_col.append(item)


diag_col = diags.columns
diag_col = diag_col[~diag_col.isin(['hadm_id'])].tolist()

AET_col = ['ab_id_count', 'ab_id_filtered_count', 'ATE', 'ATE_filtered']

MDR_col = ['MDR_before']

BI_col = ['Acinetobacter spp.', 'Enterobacteriaceae', 'Enterococcus spp.',
       'Pseudomonas aeruginosa', 'Staphylococcus aureus', 'MDR_label']

all_feat_feature = chart_col + lab_col + proc_col + med_col# + diag_col + AET_col + MDR_col
len(all_feat_feature)

163

In [ ]:
df_cate = [feat for feat in all_feat_feature if 'Cate' in feat or 'Str' in feat]
print("Number of categorical features:", len(df_cate))

df_num = [feat for feat in all_feat_feature if feat not in df_cate]
print("Number of numerical features:", len(df_num))

Number of categorical features: 56
Number of numerical features: 107


In [12]:
alc_id = all_df.ICUSTAY_ID.unique()
len(alc_id)

49023

In [13]:
alc_id[0]

30000153

In [ ]:
save_path = '/MIMIC_IV/Median_B/Final/New_All_IV_minmax/'

for i in tqdm(alc_id):
    final_df = pd.DataFrame(columns=all_feat_feature)
    chart = pd.read_csv(f'/MIMIC_IV/Final_Chart_2/{i}.csv')
    
    try:
        lab = pd.read_csv(f'/MIMIC_IV/Final_Lab_2/{i}.csv')
    except Exception as e:
        
        lab = None
        
    try:
        proc = pd.read_csv(f'/MIMIC_IV/Final_Proc_2/{i}.csv')
    except Exception as e:
       
        proc = None
        
    try:
        med = pd.read_csv(f'/MIMIC_IV/Final_Med_2/{i}.csv')
    except Exception as e:
       
        med = None

    data_frames = [chart, lab, proc, med]
    
    data_frames = [df for df in data_frames if df is not None]
    
   
    if all(df.shape[0] == chart.shape[0] for df in data_frames):
        
        this_concat = pd.concat(data_frames, axis=1)
        
       
        final_df = pd.concat([final_df, this_concat], axis=0, ignore_index=True)
    else:
      
        print(f'Skipping {i}, row counts do not match.')
    
   
    final_df = final_df.fillna(0)
    
   
    scaler = MinMaxScaler()
    final_df[df_num] = scaler.fit_transform(final_df[df_num])

    label_encoders = {feature: LabelEncoder() for feature in df_cate}
    for feature, le in label_encoders.items():
        final_df[feature] = le.fit_transform(final_df[feature])
    
   
    final_df.to_csv(f'{save_path}{i}.csv', index=False)

100%|██████████████████████████████████████████████████████████████████████████| 49023/49023 [1:44:03<00:00,  7.85it/s]
